## Labelled-data curve

How much of the fine-tuning gain survives on less labelled data. RoBERTa-large is
refit on stratified subsets of each training partition and scored on the same
untouched test split. The 1,984 point is the existing `roberta-large` row.


### Colab Setup

In [1]:
import os
import subprocess
import sys

# local runs: the repo root is one level up. Colab chdirs there below.
sys.path.insert(0, "..")

# On Colab: clone the repo, install deps, mount Drive for results.csv. The repo is
# public, so no token. Python caches imports -- restart the runtime after any code
# change, or the clone refreshes and the old module stays loaded.
REPO = "https://github.com/IronQuant/mlds_codebase.git"
ROOT = "/content/mlds_codebase"

if "google.colab" in sys.modules:
    if os.path.isdir(ROOT):
        subprocess.run(["git", "-C", ROOT, "fetch", "-q", "origin"], check=True)
        subprocess.run(
            ["git", "-C", ROOT, "reset", "--hard", "-q", "origin/main"], check=True
        )
    else:
        subprocess.run(["git", "clone", "-q", REPO, ROOT], check=True)

    subprocess.run(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            "transformers>=4.48",
            "ftfy",
            "nltk",
            "polars",
            "fastexcel",
            "sentencepiece",
            "protobuf",
        ],
        check=True,
    )
    os.chdir(ROOT)
    sys.path.insert(0, ROOT)

    # mounting is optional: without it, results land in the Colab session
    # filesystem and are lost when the runtime ends
    try:
        from google.colab import drive

        drive.mount("/content/drive")
    except Exception:
        print(f"Drive not mounted. Results will be written to {ROOT} "
              "and lost when the session ends.")

Mounted at /content/drive


### Key Imports

In [2]:
import polars as pl
import torch

from config import RESULTS_DIR, SHAH_PLM, SHAH_SEEDS
from data.loader_twd_labelled import load_splits
from models.plm_finetune import finetune

from utils.results import already_done, save_result

OUT = RESULTS_DIR / "results.csv"
ENC = "roberta-large"
SEEDS = SHAH_SEEDS
SIZES = (125, 250, 375, 500, 750, 1000, 1250, 1500, 1750)
FORCE = False
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("results ->", OUT, "| device:", DEVICE)
if DEVICE == "cuda":
    print(torch.cuda.get_device_name(0))


results -> /content/drive/MyDrive/thesis/results.csv | device: cuda
NVIDIA A100-SXM4-40GB


### Subsample

Stratified by label, so no class drops out at the small end. The test split is
never touched, only the training half shrinks.


In [3]:
import polars as pl; print(pl.__version__)
def subsample(train, n, seed):
    """
    Subsample the training data to have approximately n examples,
    Maintaining class balance.
    Args:
        train (pl.DataFrame): The training data.
        n (int): The desired number of examples.
        seed (int): Random seed for reproducibility.

    Returns:
        pl.DataFrame: The subsampled training data.
    """
    parts = []
    # narrow first: older polars aggregates every column here and trips
    # on the loader's index columns
    train = train.select("sentence", "label")
    for g in train.partition_by("label"):
        k = max(1, round(n * len(g) / len(train)))
        parts.append(g.sample(n=min(k, len(g)), shuffle=True, seed=seed))
    return pl.concat(parts)


# class balance at the small end
train, _ = load_splits("benchmark", seed=SEEDS[0])
for n in SIZES:
    s = subsample(train, n, SEEDS[0])
    print(n, "->", len(s), "|", s["label"].value_counts().sort("label").to_dicts())


1.35.2
Seed 5768 | Train: 1984 rows  |  Test: 496 rows  |  Total: 2480
125 -> 125 | [{'label': 0, 'count': 33}, {'label': 1, 'count': 30}, {'label': 2, 'count': 62}]
250 -> 250 | [{'label': 0, 'count': 66}, {'label': 1, 'count': 61}, {'label': 2, 'count': 123}]
375 -> 374 | [{'label': 0, 'count': 98}, {'label': 1, 'count': 91}, {'label': 2, 'count': 185}]
500 -> 500 | [{'label': 0, 'count': 131}, {'label': 1, 'count': 122}, {'label': 2, 'count': 247}]
750 -> 750 | [{'label': 0, 'count': 197}, {'label': 1, 'count': 183}, {'label': 2, 'count': 370}]
1000 -> 1000 | [{'label': 0, 'count': 263}, {'label': 1, 'count': 244}, {'label': 2, 'count': 493}]
1250 -> 1250 | [{'label': 0, 'count': 328}, {'label': 1, 'count': 305}, {'label': 2, 'count': 617}]
1500 -> 1500 | [{'label': 0, 'count': 394}, {'label': 1, 'count': 366}, {'label': 2, 'count': 740}]
1750 -> 1751 | [{'label': 0, 'count': 460}, {'label': 1, 'count': 427}, {'label': 2, 'count': 864}]


### Fine-tune

`finetune()` carves 20% of what it is handed for validation, so at n=125 early
stopping runs off 25 examples. We accept that noise rather than hold the
validation set fixed, since instability at small labelled counts is part of what
this curve measures.


In [4]:
cfg = SHAH_PLM[ENC]

for n in SIZES:
    for seed in SEEDS:
        model_key = f"subset-{n}:{ENC}"
        if already_done(OUT, force=FORCE, model=model_key, corpus="twd", seed=seed):
            print(f"{model_key} seed {seed}: already done, skipping")
            continue
        train, test = load_splits("benchmark", seed=seed)
        small = subsample(train, n, seed)
        print(f"{model_key} seed {seed}: {len(small)} train rows", flush=True)
        model, tok_, metrics = finetune(
            small,
            model_name=cfg["model_name"],
            lr=cfg["lr"],
            batch_size=cfg["batch_size"],
            seed=seed,
            test_df=test.select("sentence", "label"),
            device=DEVICE,
            verbose=True,
        )
        save_result(
            OUT,
            model=model_key,
            corpus="twd",
            seed=seed,
            epochs=metrics["epochs"],
            weighted_f1=round(metrics["test_f1"], 4),
            macro_f1=round(metrics["test_macro_f1"], 4),
        )
        print(f"{model_key} seed {seed}: macro={metrics['test_macro_f1']:.4f}")
        del model, tok_
        torch.cuda.empty_cache()


subset-125:roberta-large seed 5768: already done, skipping
subset-125:roberta-large seed 78516: already done, skipping
subset-125:roberta-large seed 944601: already done, skipping
subset-250:roberta-large seed 5768: already done, skipping
subset-250:roberta-large seed 78516: already done, skipping
subset-250:roberta-large seed 944601: already done, skipping
Seed 5768 | Train: 1984 rows  |  Test: 496 rows  |  Total: 2480
subset-375:roberta-large seed 5768: 374 train rows


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


config.json:   0%|          | 0.00/482 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

...

[transformers] RobertaForSequenceClassification LOAD REPORT from: roberta-large
Key                        | Status     | 
---------------------------+------------+-
lm_head.layer_norm.weight  | UNEXPECTED | 
lm_head.dense.bias         | UNEXPECTED | 
lm_head.dense.weight       | UNEXPECTED | 
lm_head.layer_norm.bias    | UNEXPECTED | 
lm_head.bias               | UNEXPECTED | 
classifier.dense.weight    | MISSING    | 
classifier.out_proj.bias   | MISSING    | 
classifier.dense.bias      | MISSING    | 
classifier.out_proj.weight | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


    epoch  0: val CE=1.0789  acc=0.3943  wF1=0.3682  mF1=0.3711  es=0  17.4s
    epoch  1: val CE=0.7857  acc=0.6143  wF1=0.6227  mF1=0.6106  es=0  18.1s
    epoch  2: val CE=0.8856  acc=0.6400  wF1=0.6460  mF1=0.6364  es=0  17.8s
    epoch  3: val CE=0.8237  acc=0.6800  wF1=0.6858  mF1=0.6616  es=0  17.6s
    epoch  4: val CE=0.9919  acc=0.6800  wF1=0.6875  mF1=0.6645  es=0  17.7s
    epoch  5: val CE=1.0304  acc=0.6629  wF1=0.6709  mF1=0.6501  es=1  17.3s
    epoch  6: val CE=1.2776  acc=0.6343  wF1=0.6414  mF1=0.6200  es=2  17.2s
    epoch  7: val CE=1.4569  acc=0.6514  wF1=0.6587  mF1=0.6414  es=3  17.3s
    epoch  8: val CE=1.5262  acc=0.6771  wF1=0.6804  mF1=0.6578  es=4  17.2s
    epoch  9: val CE=1.5224  acc=0.7029  wF1=0.7050  mF1=0.6778  es=0  17.8s
    epoch 10: val CE=1.5898  acc=0.6714  wF1=0.6774  mF1=0.6582  es=1  17.5s
    epoch 11: val CE=1.5404  acc=0.6457  wF1=0.6525  mF1=0.6368  es=2  17.3s
    epoch 12: val CE=1.6635  acc=0.6657  wF1=0.6695  mF1=0.6531  es=3  17.3s

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: roberta-large
Key                        | Status     | 
---------------------------+------------+-
lm_head.layer_norm.weight  | UNEXPECTED | 
lm_head.dense.bias         | UNEXPECTED | 
lm_head.dense.weight       | UNEXPECTED | 
lm_head.layer_norm.bias    | UNEXPECTED | 
lm_head.bias               | UNEXPECTED | 
classifier.dense.weight    | MISSING    | 
classifier.out_proj.bias   | MISSING    | 
classifier.dense.bias      | MISSING    | 
classifier.out_proj.weight | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


    epoch  0: val CE=0.8345  acc=0.6543  wF1=0.6607  mF1=0.6509  es=0  18.0s
    epoch  1: val CE=0.7241  acc=0.7229  wF1=0.7174  mF1=0.6946  es=0  18.3s
    epoch  2: val CE=0.6435  acc=0.7629  wF1=0.7621  mF1=0.7499  es=0  18.3s
    epoch  3: val CE=0.7112  acc=0.7057  wF1=0.7093  mF1=0.7032  es=1  17.6s
    epoch  4: val CE=0.8890  acc=0.7314  wF1=0.7336  mF1=0.7258  es=2  17.7s
    epoch  5: val CE=0.9182  acc=0.7286  wF1=0.7299  mF1=0.7144  es=3  17.8s
    epoch  6: val CE=1.0485  acc=0.7229  wF1=0.7272  mF1=0.7174  es=4  17.8s
    epoch  7: val CE=1.0827  acc=0.7343  wF1=0.7341  mF1=0.7214  es=5  17.7s
    epoch  8: val CE=1.2807  acc=0.7200  wF1=0.7196  mF1=0.7029  es=6  17.5s
    epoch  9: val CE=1.0594  acc=0.7429  wF1=0.7434  mF1=0.7299  es=7  17.8s
subset-1750:roberta-large seed 944601: macro=0.7035


### Curve

The 1,984 endpoint comes from the existing `roberta-large` rows, so it is the
same run reported in the approach comparison.


In [ ]:
d = (
    pl.read_csv(OUT)
    .filter(pl.col("corpus") == "twd")
    .filter(pl.col("model").str.starts_with("subset-") | (pl.col("model") == ENC))
)
for m in sorted(d["model"].unique()):
    v = d.filter(pl.col("model") == m)["macro_f1"]
    print(f"{m:26s} mean {v.mean():.4f}  sd {v.std(ddof=0):.4f}  n {len(v)}")
